# Text-to-Text (TTT) — local LLM inference on Apple Silicon

Pick a **model family** (Phi-3, Llama-3, Falcon3, Qwen2.5) and a **size** (SMALL / MEDIUM / LARGE),
then run the pipeline end to end:

1. Select the model from the `MODEL_NAME` registry (`APPLE → TTT → FAMILY → SIZE`)
2. Review every family and size: parameters, memory footprint and cost per input token
3. Load the model
4. Load a prompt set (`prompts/<set>.csv`) and count input tokens
5. Generate a response to every prompt
6. Save the responses
7. Measure the real cost of the run, per prompt and in total
8. Compare SMALL / MEDIUM / LARGE of the selected family side by side

All models run on **MLX** (`mlx-lm`) on the Apple GPU as **4-bit** quantized, instruction-tuned checkpoints,
so every family and size is compared like for like.

## 1. Configuration — model registry

In [1]:
import os
from pathlib import Path

# APPLE -> TTT -> family -> size -> model
MODEL_NAME = {
    'APPLE': {
        'TTT': {
            'Phi-3': {
                'SMALL': 'Phi-3-mini-128k-instruct',     #  3.8B
                'MEDIUM': 'Phi-3-small-128k-instruct',   #  7.4B
                'LARGE': 'Phi-3-medium-128k-instruct',   # 14.0B
            },
            'Llama-3': {
                'SMALL': 'Llama-3.2-1B-Instruct',        #  1.2B
                'MEDIUM': 'Llama-3.2-3B-Instruct',       #  3.2B
                'LARGE': 'Llama-3.1-8B-Instruct',        #  8.0B
            },
            'Falcon3': {
                'SMALL': 'Falcon3-3B-Instruct',          #  3.2B
                'MEDIUM': 'Falcon3-7B-Instruct',         #  7.5B
                'LARGE': 'Falcon3-10B-Instruct',         # 10.3B
            },
            'Qwen2.5': {
                'SMALL': 'Qwen2.5-3B-Instruct',          #  3.1B
                'MEDIUM': 'Qwen2.5-7B-Instruct',         #  7.6B
                'LARGE': 'Qwen2.5-14B-Instruct',         # 14.8B
            },
            'Llama-2': {                          # base (not chat) model; 13B/70B not included
                'MEDIUM': 'Llama-2-7b-hf',               #  6.7B
            },
        },
    },
}

# Model -> 4-bit MLX checkpoint to run, and the original checkpoint it was made from
MLX_REPOS = {
    'Phi-3-mini-128k-instruct':   ('mlx-community/Phi-3-mini-128k-instruct-4bit',   'microsoft/Phi-3-mini-128k-instruct'),
    # The two Phi-3 models below are built in models/ on first use (see LOCAL_BUILDS and section 3)
    'Phi-3-small-128k-instruct':  ('models/Phi-3-small-128k-instruct-4bit',         'microsoft/Phi-3-small-128k-instruct'),
    'Phi-3-medium-128k-instruct': ('models/Phi-3-medium-128k-instruct-4bit',        'microsoft/Phi-3-medium-128k-instruct'),
    'Llama-3.2-1B-Instruct':      ('mlx-community/Llama-3.2-1B-Instruct-4bit',      'meta-llama/Llama-3.2-1B-Instruct'),
    'Llama-3.2-3B-Instruct':      ('mlx-community/Llama-3.2-3B-Instruct-4bit',      'meta-llama/Llama-3.2-3B-Instruct'),
    'Llama-3.1-8B-Instruct':      ('mlx-community/Meta-Llama-3.1-8B-Instruct-4bit', 'meta-llama/Llama-3.1-8B-Instruct'),
    'Falcon3-3B-Instruct':        ('mlx-community/Falcon3-3B-Instruct-4bit',        'tiiuae/Falcon3-3B-Instruct'),
    'Falcon3-7B-Instruct':        ('mlx-community/Falcon3-7B-Instruct-4bit',        'tiiuae/Falcon3-7B-Instruct'),
    'Falcon3-10B-Instruct':       ('mlx-community/Falcon3-10B-Instruct-4bit',       'tiiuae/Falcon3-10B-Instruct'),
    'Qwen2.5-3B-Instruct':        ('mlx-community/Qwen2.5-3B-Instruct-4bit',        'Qwen/Qwen2.5-3B-Instruct'),
    'Qwen2.5-7B-Instruct':        ('mlx-community/Qwen2.5-7B-Instruct-4bit',        'Qwen/Qwen2.5-7B-Instruct'),
    'Qwen2.5-14B-Instruct':       ('mlx-community/Qwen2.5-14B-Instruct-4bit',       'Qwen/Qwen2.5-14B-Instruct'),
    # Converted locally on first use from an ungated copy of meta-llama/Llama-2-7b-hf (identical weights)
    'Llama-2-7b-hf':              ('models/Llama-2-7b-hf-4bit',                     'NousResearch/Llama-2-7b-hf'),
}

# Models prepared locally in models/ the first time they are selected:
#   'convert':  no working MLX upload exists, so quantize the original checkpoint to 4-bit MLX
#   'fix_rope': the MLX upload's config uses the legacy rope type 'su', which transformers 5 rejects;
#               link its weights (no copy) and rename the type to 'longrope' (the same method)
LOCAL_BUILDS = {
    'Phi-3-small-128k-instruct':  ('convert', 'microsoft/Phi-3-small-128k-instruct'),
    'Phi-3-medium-128k-instruct': ('fix_rope', 'mlx-community/Phi-3-medium-128k-instruct-4bit'),
    'Llama-2-7b-hf':              ('convert', 'NousResearch/Llama-2-7b-hf'),
}

# Phi-3 ends each chat turn with <|end|>, but these MLX uploads don't list it as a stop token,
# so without this the model keeps writing new turns until MAX_NEW_TOKENS
EXTRA_STOP_TOKENS = {
    'Phi-3-mini-128k-instruct': ['<|end|>'],
    'Phi-3-medium-128k-instruct': ['<|end|>'],
}

# Phi-3-small ships its own tokenizer code (tiktoken-based), which must be allowed to run
TRUST_REMOTE_CODE = {'Phi-3-small-128k-instruct'}

PLATFORM, TASK = 'APPLE', 'TTT'
# Environment variables TTT_FAMILY / TTT_SIZE override these, for running from the terminal
FAMILY = os.environ.get('TTT_FAMILY', 'Llama-3')   # 'Phi-3' | 'Llama-3' | 'Falcon3' | 'Qwen2.5' | 'Llama-2'
SIZE = os.environ.get('TTT_SIZE', 'MEDIUM')         # 'SMALL' | 'MEDIUM' | 'LARGE'

model_name = MODEL_NAME[PLATFORM][TASK][FAMILY][SIZE]
repo, source_repo = MLX_REPOS[model_name]

# Prompt set: a CSV with columns id,prompt in prompts/. TTT_PROMPTS overrides it from the terminal.
PROMPT_SET = os.environ.get('TTT_PROMPTS', 'ttt_prompts_500w')   # a CSV in prompts/ with columns id,prompt
INPUT_PATH = Path('prompts') / f'{PROMPT_SET}.csv'
OUTPUT_DIR = Path('outputs/ttt') / FAMILY   # outputs/ttt/<family>/<params>/<prompt set>/ (see out_dir, section 2)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_NEW_TOKENS = min(int(os.environ.get('TTT_MAX_TOKENS', '1024')), 1024)   # hard cap: no response exceeds 1024 output tokens
POWER_SAMPLE_MS = 100            # macmon sampling interval (its fastest is ~110 ms)
ELECTRICITY_USD_PER_KWH = 0.15   # your electricity price, for the $ per 1M tokens figures
RUN_FAMILY_COMPARISON = os.environ.get('TTT_COMPARE', '1') == '1'   # section 8: also run the other two sizes of FAMILY

print(f'{PLATFORM} / {TASK} / {FAMILY} / {SIZE} -> {model_name} -> {repo}')
print(f'Prompt set: {INPUT_PATH}  |  max new tokens: {MAX_NEW_TOKENS}')

APPLE / TTT / Qwen2.5 / LARGE -> Qwen2.5-14B-Instruct -> mlx-community/Qwen2.5-14B-Instruct-4bit
Prompt set: prompts/ttt_prompts_500w.csv  |  max new tokens: 1024


## 2. Model families — sizes, memory and input token cost

Architecture numbers come from each checkpoint's `config.json`; parameter counts from the original
checkpoint's safetensors metadata on Hugging Face.

| Column | Formula |
|---|---|
| Weights bf16 / 8-bit / 4-bit | params × 2 bytes / × 8.5 bits / × 4.5 bits (MLX stores a scale and bias per 64 weights, adding 0.5 bits) |
| 4-bit download | actual size of the MLX checkpoint this notebook runs |
| KV cache per input token | 2 (K and V) × layers × KV heads × head dim × 2 bytes |
| Memory at full context | 4-bit weights + KV cache × context window |
| Max context on this Mac | how many tokens fit in the GPU memory budget after loading the 4-bit weights |

**What "input token cost" means for a local model.** These are open-weight models running on your Mac, so
there is **no per-token fee ($0)**. What each input token actually costs you is:

- **Memory:** every prompt token stays in the KV cache until generation finishes (the KV column).
- **Time:** the prompt must be processed before the first output token appears (measured in section 7).
- **Context:** each token uses up part of the model's fixed context window.
- **Energy:** the GPU draws power while it reads the prompt. Section 7 measures this in millijoules per input
  token and converts it to $ at `ELECTRICITY_USD_PER_KWH`.

Families using grouped-query attention with few KV heads (Qwen2.5, Falcon3) store far less per token than
Phi-3-mini, which has one KV head per attention head.

In [2]:
import mlx.core as mx
import pandas as pd

GB = 1e9
DEVICE = mx.device_info()
GPU_BUDGET_GB = DEVICE['max_recommended_working_set_size'] / GB  # GPU memory macOS allows by default

SPECS = pd.DataFrame([
    # model,                        params,          layers, heads, kv_heads, head_dim, context, download_gb, gated_original
    ('Phi-3-mini-128k-instruct',    3_821_079_552,   32,     32,    32,       96,       131_072, 2.15,        'no'),
    ('Phi-3-small-128k-instruct',   7_392_272_384,   32,     32,    8,        128,      131_072, 4.16,        'no'),
    ('Phi-3-medium-128k-instruct',  13_960_238_080,  40,     40,    10,       128,      131_072, 7.85,        'no'),
    ('Llama-3.2-1B-Instruct',       1_235_814_400,   16,     32,    8,        64,       131_072, 0.70,        'yes'),
    ('Llama-3.2-3B-Instruct',       3_212_749_824,   28,     24,    8,        128,      131_072, 1.81,        'yes'),
    ('Llama-3.1-8B-Instruct',       8_030_261_248,   32,     32,    8,        128,      131_072, 4.52,        'yes'),
    ('Falcon3-3B-Instruct',         3_227_655_168,   22,     12,    4,        256,      32_768,  1.82,        'no'),
    ('Falcon3-7B-Instruct',         7_455_550_464,   28,     12,    4,        256,      32_768,  4.19,        'no'),
    ('Falcon3-10B-Instruct',        10_305_653_760,  40,     12,    4,        256,      32_768,  5.80,        'no'),
    ('Qwen2.5-3B-Instruct',         3_085_938_688,   36,     16,    2,        128,      32_768,  1.74,        'no'),
    ('Qwen2.5-7B-Instruct',         7_615_616_512,   28,     28,    4,        128,      32_768,  4.28,        'no'),
    ('Qwen2.5-14B-Instruct',        14_770_033_664,  48,     40,    8,        128,      32_768,  8.31,        'no'),
    ('Llama-2-7b-hf',               6_738_415_616,   32,     32,    32,       128,      4_096,   3.79,        'yes'),
], columns=['model', 'params', 'layers', 'heads', 'kv_heads', 'head_dim', 'context', 'download_gb', 'gated']).set_index('model')

SPECS['kv_bytes_per_token'] = 2 * SPECS.layers * SPECS.kv_heads * SPECS.head_dim * 2
SPECS['q4_gb'] = SPECS.params * 4.5 / 8 / GB
SPECS['full_ctx_gb'] = SPECS.q4_gb + SPECS.kv_bytes_per_token * SPECS.context / GB
SPECS['max_ctx_here'] = ((GPU_BUDGET_GB * 0.9 - SPECS.q4_gb) * GB / SPECS.kv_bytes_per_token).clip(upper=SPECS.context).astype(int)

# Output subfolder per model, named by its parameter count: outputs/ttt/<family>/<params>/<prompt set>/
PARAM_FOLDER = {
    'Phi-3-mini-128k-instruct': '3.8B', 'Phi-3-small-128k-instruct': '7B', 'Phi-3-medium-128k-instruct': '14B',
    'Llama-3.2-1B-Instruct': '1B', 'Llama-3.2-3B-Instruct': '3B', 'Llama-3.1-8B-Instruct': '8B',
    'Falcon3-3B-Instruct': '3B', 'Falcon3-7B-Instruct': '7B', 'Falcon3-10B-Instruct': '10B',
    'Qwen2.5-3B-Instruct': '3B', 'Qwen2.5-7B-Instruct': '7B', 'Qwen2.5-14B-Instruct': '14B',
    'Llama-2-7b-hf': '7B',
}

def out_dir(name):
    d = OUTPUT_DIR / PARAM_FOLDER[name] / PROMPT_SET
    d.mkdir(parents=True, exist_ok=True)
    return d

rows = []
for family, sizes in MODEL_NAME[PLATFORM][TASK].items():
    for size, name in sizes.items():
        s = SPECS.loc[name]
        rows.append({
            '': '▶' if name == model_name else '',
            'Family': family,
            'Size': size,
            'Model': name,
            'Params (B)': f'{s.params / 1e9:.2f}',
            'Context': f'{s.context:,}',
            'Weights bf16 (GB)': f'{s.params * 2 / GB:.1f}',
            'Weights 8-bit (GB)': f'{s.params * 8.5 / 8 / GB:.1f}',
            'Weights 4-bit (GB)': f'{s.q4_gb:.1f}',
            '4-bit download (GB)': f'{s.download_gb:.2f}',
            'KV cache / input token (KB)': f'{s.kv_bytes_per_token / 1024:.0f}',
            'Memory at full context, 4-bit (GB)': f'{s.full_ctx_gb:.1f}',
            'Max context on this Mac': f'{s.max_ctx_here:,}',
            'API cost / input token': '$0 (local)',
            'Original gated on HF': s.gated,
        })

print(f"{DEVICE['device_name']}: {DEVICE['memory_size'] / GB:.0f} GB unified memory, "
      f"{GPU_BUDGET_GB:.1f} GB usable by the GPU by default.  ▶ = selected model")
pd.DataFrame(rows).style.hide(axis='index')

Apple M5 Max: 52 GB unified memory, 40.2 GB usable by the GPU by default.  ▶ = selected model


,Family,Size,Model,Params (B),Context,Weights bf16 (GB),Weights 8-bit (GB),Weights 4-bit (GB),4-bit download (GB),KV cache / input token (KB),"Memory at full context, 4-bit (GB)",Max context on this Mac,API cost / input token,Original gated on HF
,Phi-3,SMALL,Phi-3-mini-128k-instruct,3.82,"131,072",7.6,4.1,2.1,2.15,384,53.7,"86,546",$0 (local),no
,Phi-3,MEDIUM,Phi-3-small-128k-instruct,7.39,"131,072",14.8,7.9,4.2,4.16,128,21.3,"131,072",$0 (local),no
,Phi-3,LARGE,Phi-3-medium-128k-instruct,13.96,"131,072",27.9,14.8,7.9,7.85,200,34.7,"131,072",$0 (local),no
,Llama-3,SMALL,Llama-3.2-1B-Instruct,1.24,"131,072",2.5,1.3,0.7,0.70,32,5.0,"131,072",$0 (local),yes
,Llama-3,MEDIUM,Llama-3.2-3B-Instruct,3.21,"131,072",6.4,3.4,1.8,1.81,112,16.8,"131,072",$0 (local),yes
,Llama-3,LARGE,Llama-3.1-8B-Instruct,8.03,"131,072",16.1,8.5,4.5,4.52,128,21.7,"131,072",$0 (local),yes
,Falcon3,SMALL,Falcon3-3B-Instruct,3.23,"32,768",6.5,3.4,1.8,1.82,88,4.8,"32,768",$0 (local),no
,Falcon3,MEDIUM,Falcon3-7B-Instruct,7.46,"32,768",14.9,7.9,4.2,4.19,112,8.0,"32,768",$0 (local),no
,Falcon3,LARGE,Falcon3-10B-Instruct,10.31,"32,768",20.6,10.9,5.8,5.80,160,11.2,"32,768",$0 (local),no
,Qwen2.5,SMALL,Qwen2.5-3B-Instruct,3.09,"32,768",6.2,3.3,1.7,1.74,36,2.9,"32,768",$0 (local),no


The Llama originals on Hugging Face need an approved access request, but the `mlx-community` 4-bit
versions used here download without one. To try other precisions, swap a repo in `MLX_REPOS` for its
`-8bit` or bf16 variant, or convert an original yourself with
`mlx_lm.convert --hf-path <original repo> --mlx-path <folder> -q --q-bits 8`.

## 3. Load the model

Two Phi-3 models are prepared in `models/` the first time they are selected (see `LOCAL_BUILDS`), then load from disk:

- **Phi-3-small** has no working MLX upload, so it is converted from Microsoft's original checkpoint
  (about 15 GB download) to a 4-bit MLX model.
- **Phi-3-medium**'s MLX upload is fine, but its config uses a legacy name for its RoPE scaling type that
  transformers 5 rejects. A local copy links to the downloaded weights and fixes the name.

In [3]:
import json
import shutil
import time
from huggingface_hub import snapshot_download
from mlx.utils import tree_flatten
from mlx_lm import convert, load

def prepare_local_model(name, repo):
    method, src = LOCAL_BUILDS[name]
    dst = Path(repo)
    if method == 'convert':
        print(f'Converting {src} -> {repo} (4-bit), one-time step...')
        src_path = snapshot_download(src, token=False, allow_patterns=[
            '*.json', '*.safetensors', '*.py', '*.model', '*.tiktoken', '*.txt'])
        convert(hf_path=src_path, mlx_path=repo, quantize=True, q_bits=4, q_group_size=64,
                trust_remote_code=name in TRUST_REMOTE_CODE)
    elif method == 'fix_rope':
        print(f'Building {repo} from {src} with the rope type fixed, one-time step...')
        snap = Path(snapshot_download(src))
        dst.mkdir(parents=True)
        for f in snap.iterdir():
            if f.suffix == '.safetensors':
                (dst / f.name).symlink_to(f.resolve())  # link to the Hugging Face cache, no copy
            elif f.name != 'config.json':
                shutil.copy(f, dst / f.name)
        cfg = json.loads((snap / 'config.json').read_text())
        if cfg.get('rope_scaling', {}).get('type') == 'su':
            cfg['rope_scaling']['type'] = 'longrope'
        (dst / 'config.json').write_text(json.dumps(cfg, indent=2))

def load_mlx_model(name):
    repo, _ = MLX_REPOS[name]
    trust = name in TRUST_REMOTE_CODE
    if name in LOCAL_BUILDS and not Path(repo).exists():
        prepare_local_model(name, repo)
    t0 = time.perf_counter()
    model, tokenizer = load(repo, tokenizer_config={'trust_remote_code': True} if trust else None)
    template = Path(repo) / 'chat_template.jinja'
    if tokenizer.chat_template is None and template.exists():
        # Custom tokenizers (Phi-3-small) don't read the template file saved by mlx_lm.convert
        tokenizer._tokenizer.chat_template = template.read_text()
    for token in EXTRA_STOP_TOKENS.get(name, []):
        tokenizer.add_eos_token(token)
    load_s = time.perf_counter() - t0
    n_params = sum(v.size for _, v in tree_flatten(model.parameters()))
    return model, tokenizer, load_s, n_params

model, tokenizer, load_seconds, n_stored = load_mlx_model(model_name)
spec = SPECS.loc[model_name]
print(f'Loaded {repo} in {load_seconds:.1f}s on {mx.default_device()}')
print(f'Parameters: {spec.params / 1e9:.2f}B  |  layers {spec.layers}, heads {spec.heads}, '
      f'KV heads {spec.kv_heads}, context {spec.context:,}')
print(f'Weights in memory: {mx.get_active_memory() / GB:.2f} GB (4-bit)')

/Users/genailab/mlx-projects/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 2389.51it/s]

Loaded mlx-community/Qwen2.5-14B-Instruct-4bit in 1.2s on Device(gpu, 0)
Parameters: 14.77B  |  layers 48, heads 40, KV heads 8, context 32,768
Weights in memory: 8.31 GB (4-bit)


## 4. Load the prompt set

Every prompt in `INPUT_PATH` is sent to the model as a separate conversation, wrapped in the model's own
chat template. A prompt that asks for a length ("In 500 words") gets that number recorded as its word target,
so section 7 can check how closely each model follows it.

In [4]:
import re

PROMPTS = pd.read_csv(INPUT_PATH, dtype={'id': str})
PROMPTS['id'] = PROMPTS['id'].str.zfill(2)
PROMPTS['words_target'] = PROMPTS.prompt.str.extract(r'[Ii]n (\d+) words', expand=False).astype('Int64')

BASE_STOP = '\n### '   # a base model starting a new section has finished its answer

def build_prompt(tokenizer, text):
    if tokenizer.chat_template is None:
        # Base (non-chat) models such as Llama-2-7b-hf have no chat template, so the request is written as a
        # document that the model continues with the answer
        prompt = f'Below is an instruction, followed by a response that completes it.\n\n### Instruction:\n{text}\n\n### Response:\n'
        return prompt, tokenizer.encode(prompt)
    messages = [{'role': 'user', 'content': text}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    return prompt, tokenizer.encode(prompt, add_special_tokens=False)

def encode_all(tokenizer):
    return [build_prompt(tokenizer, text)[1] for text in PROMPTS.prompt]

input_ids_all = encode_all(tokenizer)
n_inputs = [len(ids) for ids in input_ids_all]
print(f'Prompt set: {INPUT_PATH} ({len(PROMPTS)} prompts, {sum(n_inputs):,} input tokens with chat formatting)')
print(f'Largest prompt uses {max(n_inputs) / spec.context:.2%} of the {spec.context:,}-token context window')
pd.DataFrame({
    'id': PROMPTS.id,
    'prompt': PROMPTS.prompt.str.slice(0, 90).where(PROMPTS.prompt.str.len() <= 90, PROMPTS.prompt.str.slice(0, 87) + '...'),
    'input tokens': n_inputs,
    'word target': PROMPTS.words_target,
}).style.hide(axis='index')

Prompt set: prompts/ttt_prompts_500w.csv (10 prompts, 571 input tokens with chat formatting)
Largest prompt uses 0.27% of the 32,768-token context window


id,prompt,input tokens,word target
01,Draft a professional email to a manager requesting a recommendation for a new job oppor...,53,500
02,"Provide a brief background on the UK Prime Minister in 1995, highlighting key accomplis...",61,500
03,"Explain the current phase of the moon, its significance, and how it fits within the lun...",57,500
04,"Describe the logic used to calculate the time for baking 96 cookies, considering oven c...",57,500
05,"Imagine three friends, Mia, Noah, and Olivia, each running a race. Mia finishes ahead o...",87,500
06,Tell me about Global warming.(In 500 words),42,500
07,Tell the History of USA.(In 500 words),42,500
08,"Draft a compelling advertisement for a new grocery store in a small town, including the...",65,500
09,"Explain the basic Blackjack game, including rules, logic, and example gameplay.(In 500 ...",52,500
10,Breakdown the steering and breaking components of a Cessna 172N. (In 500 words),55,500


## 5. Generate

Greedy decoding (temperature 0), so results are reproducible.

**Power monitoring.** `macmon` (no sudo needed) runs in the background for the whole notebook, sampling the
chip's power sensors about every 110 ms. Each run's energy is split into the **prefill** window (reading the
prompt, up to the first output token) and the **decode** window (generating). Samples that straddle a window
edge are split by overlap. If `macmon` isn't installed, the power metrics are left empty and everything else runs.

Limits: macmon reports Apple's on-chip power estimates, not a wall-meter reading. Prefill often lasts under half
a second, so only a few samples cover it, and per-input-token energy is approximate.

In [5]:
import atexit
import json
import shutil
import subprocess
import threading

class PowerMonitor:
    """Runs `macmon pipe` in the background and keeps every sample.

    Each sample is the average over the interval since the previous one, so the energy in any time
    window is the sum of power x (overlap of that interval with the window).
    """
    def __init__(self, interval_ms):
        self.samples = []  # (time.perf_counter(), macmon sample dict)
        self.available = shutil.which('macmon') is not None
        if not self.available:
            print('macmon not found: power and energy metrics will be empty (install: brew install macmon)')
            return
        self.proc = subprocess.Popen(['macmon', 'pipe', '-i', str(interval_ms)],
                                     stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
        atexit.register(self.proc.terminate)
        threading.Thread(target=self._read, daemon=True).start()
        t_start = time.perf_counter()
        while len(self.samples) < 2 and time.perf_counter() - t_start < 5:  # macmon takes ~0.7 s to start
            time.sleep(0.05)
        self.available = len(self.samples) >= 2

    def _read(self):
        for line in self.proc.stdout:
            try:
                self.samples.append((time.perf_counter(), json.loads(line)))
            except json.JSONDecodeError:
                pass

    def wait_past(self, t, timeout=2.0):
        """Block until a sample covering time t has arrived."""
        deadline = time.perf_counter() + timeout
        while self.samples[-1][0] < t and time.perf_counter() < deadline:
            time.sleep(0.02)

    def window(self, a, b):
        """Samples overlapping [a, b], as (seconds of overlap, sample)."""
        return [(min(t, b) - max(t_prev, a), s)
                for (t_prev, _), (t, s) in zip(self.samples, self.samples[1:])
                if min(t, b) - max(t_prev, a) > 0]

def soc_power(s):
    return s['all_power'] + s['ram_power']  # CPU + GPU + Neural Engine + memory, in watts

def integrate(win, fn):
    return sum(overlap * fn(s) for overlap, s in win)

POWER_KEYS = ['gpu_util_avg_pct', 'gpu_util_peak_pct', 'gpu_freq_avg_mhz', 'gpu_power_avg_w', 'gpu_power_peak_w',
              'soc_power_avg_w', 'gpu_temp_max_c', 'energy_j', 'energy_net_j',
              'prefill_energy_mj_per_input_token', 'decode_energy_mj_per_output_token', 'power_samples']

def summarize_power(t0, t_first, t_end, n_input, n_output):
    if not POWER.available:
        return dict.fromkeys(POWER_KEYS)
    POWER.wait_past(t_end)
    run = POWER.window(t0, t_end)
    samples = [s for _, s in run]
    dur = t_end - t0
    e_soc = integrate(run, soc_power)
    return {
        'gpu_util_avg_pct': round(100 * integrate(run, lambda s: s['gpu_active_ratio']) / dur, 1),
        'gpu_util_peak_pct': round(100 * max(s['gpu_active_ratio'] for s in samples), 1),
        'gpu_freq_avg_mhz': round(integrate(run, lambda s: s['gpu_freq_mhz']) / dur),
        'gpu_power_avg_w': round(integrate(run, lambda s: s['gpu_power']) / dur, 1),
        'gpu_power_peak_w': round(max(s['gpu_power'] for s in samples), 1),
        'soc_power_avg_w': round(e_soc / dur, 1),
        'gpu_temp_max_c': round(max(s['temp']['gpu_temp_avg'] for s in samples), 1),
        'energy_j': round(e_soc, 2),
        'energy_net_j': round(e_soc - IDLE_SOC_W * dur, 2),
        'prefill_energy_mj_per_input_token': round(1000 * integrate(POWER.window(t0, t_first), soc_power) / n_input, 3),
        'decode_energy_mj_per_output_token': round(1000 * integrate(POWER.window(t_first, t_end), soc_power) / max(n_output, 1), 2),
        'power_samples': len(run),
    }

POWER = PowerMonitor(POWER_SAMPLE_MS)
if POWER.available:
    time.sleep(1.0)  # idle baseline, measured with the model loaded but not running
    a, b = POWER.samples[0][0], POWER.samples[-1][0]
    IDLE_SOC_W = integrate(POWER.window(a, b), soc_power) / (b - a)
    print(f'Power monitor running (macmon, ~{POWER_SAMPLE_MS} ms samples). Idle chip power: {IDLE_SOC_W:.2f} W')

def cpu_clusters():
    """CPU cluster names and core counts from macOS (e.g. M5 Max: 6 'Super' + 12 'Performance' cores)."""
    def sysctl(key):
        return subprocess.run(['sysctl', '-n', key], capture_output=True, text=True).stdout.strip()
    return [(sysctl(f'hw.perflevel{i}.name') or f'level{i}', int(sysctl(f'hw.perflevel{i}.logicalcpu') or 0)) for i in (0, 1)]

CPU_CLUSTERS = cpu_clusters()
HARDWARE = {}   # model name -> hardware_metrics() for its whole run

def hardware_metrics(t0, t1):
    """GPU, CPU, Neural Engine (NPU), memory and whole-chip activity over [t0, t1], from the macmon samples.

    macmon calls the two CPU clusters 'pcpu' (perflevel0) and 'ecpu' (perflevel1); they are reported here
    under the names macOS gives them.
    """
    if not POWER.available:
        return None
    POWER.wait_past(t1)
    win = POWER.window(t0, t1)
    samples = [s for _, s in win]
    dur = t1 - t0
    avg = lambda fn: integrate(win, fn) / dur
    peak = lambda fn: max(fn(s) for s in samples)
    energy = lambda fn: round(integrate(win, fn), 2)
    pct = lambda x: round(100 * x, 1)
    (c0_name, c0_cores), (c1_name, c1_cores) = CPU_CLUSTERS
    c0, c1 = c0_name.lower(), c1_name.lower()
    ane = lambda s: s['ane_power']
    return {
        'gpu': {
            'avg_utilization_pct': pct(avg(lambda s: s['gpu_active_ratio'])),
            'peak_utilization_pct': pct(peak(lambda s: s['gpu_active_ratio'])),
            'avg_frequency_mhz': round(avg(lambda s: s['gpu_freq_mhz'])),
            'max_frequency_mhz': round(peak(lambda s: s['gpu_freq_mhz'])),
            'avg_power_watts': round(avg(lambda s: s['gpu_power']), 1),
            'peak_power_watts': round(peak(lambda s: s['gpu_power']), 1),
            'energy_joules': energy(lambda s: s['gpu_power']),
            'avg_temperature_c': round(avg(lambda s: s['temp']['gpu_temp_avg']), 1),
            'max_temperature_c': round(peak(lambda s: s['temp']['gpu_temp_avg']), 1),
        },
        'cpu': {
            'avg_utilization_pct': pct(avg(lambda s: s['cpu_active_ratio'])),
            'peak_utilization_pct': pct(peak(lambda s: s['cpu_active_ratio'])),
            f'{c0}_cores': c0_cores,
            f'{c0}_cores_avg_utilization_pct': pct(avg(lambda s: s['pcpu_active_ratio'])),
            f'{c0}_cores_avg_frequency_mhz': round(avg(lambda s: s['pcpu_freq_mhz'])),
            f'{c1}_cores': c1_cores,
            f'{c1}_cores_avg_utilization_pct': pct(avg(lambda s: s['ecpu_active_ratio'])),
            f'{c1}_cores_avg_frequency_mhz': round(avg(lambda s: s['ecpu_freq_mhz'])),
            'avg_power_watts': round(avg(lambda s: s['cpu_power']), 1),
            'peak_power_watts': round(peak(lambda s: s['cpu_power']), 1),
            'energy_joules': energy(lambda s: s['cpu_power']),
            'avg_temperature_c': round(avg(lambda s: s['temp']['cpu_temp_avg']), 1),
            'max_temperature_c': round(peak(lambda s: s['temp']['cpu_temp_avg']), 1),
        },
        'npu': {
            'name': 'Apple Neural Engine (ANE)',
            'used': peak(ane) > 0.05,
            'avg_power_watts': round(avg(ane), 2),
            'peak_power_watts': round(peak(ane), 2),
            'energy_joules': energy(ane),
            'note': 'MLX runs models on the GPU, so the Neural Engine stays idle (about 0 W) during these runs.',
        },
        'system_memory': {
            'avg_power_watts': round(avg(lambda s: s['ram_power']), 1),
            'energy_joules': energy(lambda s: s['ram_power']),
            'ram_total_gb': round(samples[0]['memory']['ram_total'] / 1e9, 1),
            'avg_ram_used_gb': round(avg(lambda s: s['memory']['ram_usage']) / 1e9, 1),
            'peak_ram_used_gb': round(peak(lambda s: s['memory']['ram_usage']) / 1e9, 1),
            'peak_swap_used_gb': round(peak(lambda s: s['memory']['swap_usage']) / 1e9, 2),
            'note': 'RAM used is for the whole Mac, including other apps, not just this model.',
        },
        'whole_chip': {
            'avg_power_watts': round(avg(soc_power), 1),
            'peak_power_watts': round(peak(soc_power), 1),
            'energy_joules': energy(soc_power),
            'idle_power_watts': round(IDLE_SOC_W, 2),
            'note': 'CPU + GPU + Neural Engine + memory.',
        },
        'measured_seconds': round(dur, 2),
        'power_samples': len(win),
    }

Power monitor running (macmon, ~100 ms samples). Idle chip power: 0.35 W


### Per-prompt metrics analysis

Every prompt is wrapped in `metrics_monitor_start()` / `metrics_monitor_stop()` from `macmon_metrics/metrics_monitor.py`,
which records a macmon time series for that prompt. Results go to a separate folder,
`metrics_analysis/ttt/<family>/<params>/<prompt set>/`:

- `System_Level_Metrics/<prompt id>_system_metrics.csv`: CPU / GPU / NPU utilisation, power and temperature, RAM used
  and total system power, sampled every 100 ms while that prompt ran
- `Model_Level_Metrics/<model>_prompt_metrics.csv`: one row per prompt with input tokens, output tokens, the first
  token generated and its latency (time to first token), inference rate (tokens/s) and inference time

Section 9 combines every model's `Model_Level_Metrics` CSV into `metrics_analysis/ttt/all_models_prompt_metrics_<prompt set>.csv`.

In [6]:
import macmon_metrics.metrics_monitor as mm
from macmon_metrics.metrics_monitor import metrics_monitor_start, metrics_monitor_stop, setup_output_directory
from mlx_lm import stream_generate
from mlx_lm.sample_utils import make_sampler

METRICS_ROOT = Path('metrics_analysis/ttt')

def analysis_dirs(name):
    """(System_Level_Metrics, Model_Level_Metrics) folders for one model and prompt set."""
    return setup_output_directory(str(METRICS_ROOT / FAMILY / PARAM_FOLDER[name] / PROMPT_SET))

# Start the macmon tracker now: it takes about a second to produce its first sample, and starting it here means
# every prompt's capture has samples from its first moment.
if mm._tracker is None:
    mm._tracker = mm.MacMonTracker(binary_path=mm._resolve_macmon_binary())
    mm._tracker.start()
    time.sleep(1.5)

def generate(model, tokenizer, input_ids):
    mx.reset_peak_memory()
    text = ''
    t_first = None
    first_token = None
    t0 = time.perf_counter()
    for r in stream_generate(model, tokenizer, input_ids, max_tokens=MAX_NEW_TOKENS, sampler=make_sampler(temp=0.0)):
        if t_first is None:
            t_first = time.perf_counter()  # first token arrives once the prompt has been processed
            first_token = r.text
        text += r.text
        if tokenizer.chat_template is None and BASE_STOP in text.lstrip():
            text = text.lstrip().split(BASE_STOP)[0]
            break
    t_end = time.perf_counter()
    return {
        'text': text,
        'total_s': t_end - t0,
        'output_tokens': r.generation_tokens,
        'prefill_s': r.prompt_tokens / r.prompt_tps,
        'decode_tps': r.generation_tps,
        'peak_gb': r.peak_memory,
        'finish_reason': r.finish_reason or 'stop',   # None when stopped early at BASE_STOP
        'ttft_s': t_first - t0,
        'first_token': first_token,
        'power': summarize_power(t0, t_first, t_end, len(input_ids), r.generation_tokens),
    }

def warm_up(model, tokenizer):
    """One short generation so GPU kernels are compiled before anything is timed."""
    for _ in stream_generate(model, tokenizer, tokenizer.encode('Hello'), max_tokens=8):
        pass

def save_prompt_metrics(name, runs, folder):
    """Model_Level_Metrics CSV: one row per prompt."""
    size = next(sz for sizes in MODEL_NAME[PLATFORM][TASK].values() for sz, n in sizes.items() if n == name)
    rows = [{
        'prompt_id': g['id'],
        'model': name,
        'family': FAMILY,
        'size': size,
        'prompt': g['prompt'],
        'input_tokens': g['input_tokens'],
        'output_tokens': g['output_tokens'],
        'output_words': g['words'],
        'first_token': json.dumps(g['first_token'])[1:-1],   # escaped, so newlines/spaces stay visible in the CSV
        'time_to_first_token (ms)': round(1000 * g['ttft_s'], 1),
        'prompt_processing_rate (tokens/s)': round(g['input_tokens'] / g['prefill_s'], 1),
        'inference_rate (tokens/s)': round(g['decode_tps'], 2),
        'inference_time (s)': round(g['total_s'], 3),
        'response': g['text'],
    } for g in runs]
    path = Path(folder) / f'{name}_prompt_metrics.csv'
    pd.DataFrame(rows).to_csv(path, index=False)
    return path

def run_prompt_set(model, tokenizer, name):
    warm_up(model, tokenizer)
    sys_dir, model_dir = analysis_dirs(name)
    runs = []
    t_start = time.perf_counter()
    for row, ids in zip(PROMPTS.itertuples(index=False), encode_all(tokenizer)):
        metrics_monitor_start(idx=row.id, metrics_folder=sys_dir)
        is_plugged = mm._tracker.capture_is_plugged
        g = generate(model, tokenizer, ids)
        metrics_monitor_stop()
        g['is_plugged'] = is_plugged
        g['text'] = g['text'].strip()
        g.update(id=row.id, prompt=row.prompt, input_tokens=len(ids), words=len(g['text'].split()),
                 words_target=None if pd.isna(row.words_target) else int(row.words_target))
        runs.append(g)
        print(f"  [{row.id}] {len(ids):>5} in -> {g['output_tokens']:>4} out, {g['words']:>4} words, "
              f"{g['decode_tps']:5.1f} tok/s, {g['total_s']:5.1f}s ({g['finish_reason']})")
    HARDWARE[name] = hardware_metrics(t_start, time.perf_counter())
    print(f'Per-prompt metrics: {save_prompt_metrics(name, runs, model_dir)} (+ {len(runs)} system CSVs in {sys_dir})')
    return runs

print(f'Running {len(PROMPTS)} prompts on {model_name}...')
runs = run_prompt_set(model, tokenizer, model_name)
print(f"Done: {sum(g['output_tokens'] for g in runs):,} tokens in {sum(g['total_s'] for g in runs):.1f}s")

Running 10 prompts on Qwen2.5-14B-Instruct...


  [01]    53 in ->  417 out,  367 words,  62.0 tok/s,   6.9s (stop)


  [02]    61 in ->  646 out,  511 words,  61.7 tok/s,  10.6s (stop)


  [03]    57 in ->  689 out,  567 words,  61.7 tok/s,  11.3s (stop)


  [04]    57 in ->  641 out,  508 words,  61.6 tok/s,  10.6s (stop)


...

  [06]    42 in ->  591 out,  500 words,  61.6 tok/s,   9.7s (stop)


  [07]    42 in ->  869 out,  655 words,  61.3 tok/s,  14.3s (stop)


  [08]    65 in ->  664 out,  511 words,  61.5 tok/s,  11.0s (stop)


  [09]    52 in ->  836 out,  607 words,  61.2 tok/s,  13.8s (stop)


  [10]    55 in ->  914 out,  684 words,  61.1 tok/s,  15.1s (stop)
Per-prompt metrics: metrics_analysis/ttt/Qwen2.5/14B/ttt_prompts_500w/Model_Level_Metrics/Qwen2.5-14B-Instruct_prompt_metrics.csv (+ 10 system CSVs in metrics_analysis/ttt/Qwen2.5/14B/ttt_prompts_500w/System_Level_Metrics)
Done: 6,908 tokens in 114.0s


## 6. Save the responses

One text file per prompt: `outputs/ttt/<family>/<params>/<prompt set>/<model>_response_<id>.txt`.

In [7]:
import textwrap

def save_responses(name, runs):
    d = out_dir(name)
    for g in runs:
        (d / f"{name}_response_{g['id']}.txt").write_text(g['text'] + '\n')
    return d

saved_to = save_responses(model_name, runs)
print(f'Saved {len(runs)} responses to {saved_to}/\n')
first = runs[0]
print(f"=== Prompt {first['id']}: {textwrap.shorten(first['prompt'], 200)}\n")
print(textwrap.fill(first['text'][:1500], width=100, replace_whitespace=False) + (' ...' if len(first['text']) > 1500 else ''))

Saved 10 responses to outputs/ttt/Qwen2.5/14B/ttt_prompts_500w/

=== Prompt 01: Draft a professional email to a manager requesting a recommendation for a new job opportunity. (In 500 words)

Subject: Request for Professional Recommendation for Upcoming Job Opportunity

Dear [Manager's
Name],

I hope this email finds you well. I am writing to request your assistance in providing a
professional recommendation for a job opportunity that I am currently pursuing. As one of my former
managers, I believe your endorsement would greatly enhance my application and provide valuable
insight into my professional capabilities and character.

Over the past [duration of employment], I
have had the privilege of working under your guidance and have learned immensely from your
leadership. Your mentorship has been instrumental in shaping my professional journey and has
significantly contributed to my personal and professional growth. I am now at a stage where I am
seeking to further my career in a role th

## 7. Cost of this run

The input-token costs from section 2 and the energy it took, measured per prompt and totalled over the set.
Electricity cost = energy per token × `ELECTRICITY_USD_PER_KWH` (1 kWh = 3.6 MJ).
**Hit limit** means the response was cut off at `MAX_NEW_TOKENS`.

In [8]:
def usd_per_1m(mj_per_token):
    """Electricity cost of one million tokens: mJ/token -> kWh (1 kWh = 3.6e9 mJ) -> $."""
    if mj_per_token is None:
        return None
    return round(mj_per_token * 1e6 / 3.6e9 * ELECTRICITY_USD_PER_KWH, 5)

def run_metrics(name, n_input, gen, load_s):
    s = SPECS.loc[name]
    kv = int(s.kv_bytes_per_token)
    return {
        'family': next(f for f, sz in MODEL_NAME[PLATFORM][TASK].items() if name in sz.values()),
        'model': name,
        'repo': MLX_REPOS[name][0],
        'params_b': round(s.params / 1e9, 2),
        'load_s': round(load_s, 1),
        'input_tokens': n_input,
        'output_tokens': gen['output_tokens'],
        'prefill_s': round(gen['prefill_s'], 2),
        'prefill_ms_per_input_token': round(gen['prefill_s'] * 1000 / n_input, 3),
        'prefill_tok_per_s': round(n_input / gen['prefill_s']),
        'decode_tok_per_s': round(gen['decode_tps'], 1),
        'kv_kb_per_token': round(kv / 1024),
        'kv_cache_mb_this_prompt': round(kv * (n_input + gen['output_tokens']) / 1e6, 1),
        'context_used_pct': round(100 * (n_input + gen['output_tokens']) / s.context, 2),
        'peak_memory_gb': round(gen['peak_gb'], 2),
        'api_cost_usd': 0.0,
        **gen['power'],
        'usd_per_1m_input_tokens': usd_per_1m(gen['power']['prefill_energy_mj_per_input_token']),
        'usd_per_1m_output_tokens': usd_per_1m(gen['power']['decode_energy_mj_per_output_token']),
        'electricity_usd_per_kwh': ELECTRICITY_USD_PER_KWH,
    }

def weighted(per, key, weight):
    """Average of per-prompt values weighted by another column (None if power wasn't measured)."""
    pairs = [(p[key], p[weight]) for p in per if p[key] is not None]
    total = sum(w for _, w in pairs)
    return round(sum(v * w for v, w in pairs) / total, 3) if pairs and total else None

def set_metrics(name, runs, load_s):
    per = []
    for g in runs:
        m = run_metrics(name, g['input_tokens'], g, load_s)
        per.append({'id': g['id'], 'words': g['words'], 'words_target': g['words_target'],
                    'finish_reason': g['finish_reason'], 'total_s': round(g['total_s'], 2),
                    **{k: v for k, v in m.items() if k not in
                       ('family', 'model', 'repo', 'params_b', 'load_s', 'api_cost_usd', 'electricity_usd_per_kwh')}})
    s = SPECS.loc[name]
    tot_in = sum(p['input_tokens'] for p in per)
    tot_out = sum(p['output_tokens'] for p in per)
    prefill_s = sum(g['prefill_s'] for g in runs)
    decode_s = sum(g['output_tokens'] / g['decode_tps'] for g in runs)
    measured = per[0]['energy_j'] is not None
    targets = [p for p in per if p['words_target']]
    summary = {
        'family': next(f for f, sz in MODEL_NAME[PLATFORM][TASK].items() if name in sz.values()),
        'model': name,
        'repo': MLX_REPOS[name][0],
        'params_b': round(s.params / 1e9, 2),
        'prompt_set': PROMPT_SET,
        'prompts': len(per),
        'load_s': round(load_s, 1),
        'total_input_tokens': tot_in,
        'total_output_tokens': tot_out,
        'total_time_s': round(sum(p['total_s'] for p in per), 1),
        'avg_words': round(sum(p['words'] for p in per) / len(per)),
        'avg_words_vs_target_pct': round(100 * sum(p['words'] / p['words_target'] for p in targets) / len(targets)) if targets else None,
        'prompts_hit_token_limit': sum(p['finish_reason'] == 'length' for p in per),
        'prefill_tok_per_s': round(tot_in / prefill_s),
        'decode_tok_per_s': round(tot_out / decode_s, 1),
        'peak_memory_gb': max(p['peak_memory_gb'] for p in per),
        'kv_kb_per_token': per[0]['kv_kb_per_token'],
        'api_cost_usd': 0.0,
        'gpu_util_avg_pct': weighted(per, 'gpu_util_avg_pct', 'total_s'),
        'gpu_power_avg_w': weighted(per, 'gpu_power_avg_w', 'total_s'),
        'gpu_power_peak_w': max(p['gpu_power_peak_w'] for p in per) if measured else None,
        'soc_power_avg_w': weighted(per, 'soc_power_avg_w', 'total_s'),
        'gpu_temp_max_c': max(p['gpu_temp_max_c'] for p in per) if measured else None,
        'energy_j': round(sum(p['energy_j'] for p in per), 1) if measured else None,
        'energy_net_j': round(sum(p['energy_net_j'] for p in per), 1) if measured else None,
        'prefill_energy_mj_per_input_token': weighted(per, 'prefill_energy_mj_per_input_token', 'input_tokens'),
        'decode_energy_mj_per_output_token': weighted(per, 'decode_energy_mj_per_output_token', 'output_tokens'),
        'electricity_usd_per_kwh': ELECTRICITY_USD_PER_KWH,
    }
    summary['usd_per_1m_input_tokens'] = usd_per_1m(summary['prefill_energy_mj_per_input_token'])
    summary['usd_per_1m_output_tokens'] = usd_per_1m(summary['decode_energy_mj_per_output_token'])
    return {'summary': summary, 'per_prompt': per}

result = set_metrics(model_name, runs, load_seconds)
sm = result['summary']

rows = [
    ('Prompts', f"{sm['prompts']} ({sm['prompts_hit_token_limit']} hit the {MAX_NEW_TOKENS}-token limit)"),
    ('Input / output tokens (total)', f"{sm['total_input_tokens']:,} / {sm['total_output_tokens']:,}"),
    ('Total generation time', f"{sm['total_time_s']} s"),
    ('Average response length', f"{sm['avg_words']} words" + (f" ({sm['avg_words_vs_target_pct']}% of target)" if sm['avg_words_vs_target_pct'] else '')),
    ('Prefill speed / decode speed', f"{sm['prefill_tok_per_s']:,} / {sm['decode_tok_per_s']} tok/s"),
    ('Memory per token (KV cache)', f"{sm['kv_kb_per_token']} KB"),
    ('Peak memory (weights + cache)', f"{sm['peak_memory_gb']} GB"),
    ('API cost', '$0.00 (runs locally)'),
]
if sm['energy_j'] is not None:
    rows += [
        ('GPU utilisation (avg) / GPU power (avg, peak)', f"{sm['gpu_util_avg_pct']:.1f}% / {sm['gpu_power_avg_w']:.1f} W, {sm['gpu_power_peak_w']} W"),
        ('Chip power, avg (CPU + GPU + memory)', f"{sm['soc_power_avg_w']:.1f} W"),
        ('GPU temperature (max)', f"{sm['gpu_temp_max_c']} °C"),
        ('Energy for the whole set', f"{sm['energy_j']} J ({sm['energy_net_j']} J above idle)"),
        ('Energy per input / output token', f"{sm['prefill_energy_mj_per_input_token']:.1f} / {sm['decode_energy_mj_per_output_token']:.1f} mJ"),
        (f'Electricity per 1M input / output tokens (at ${ELECTRICITY_USD_PER_KWH}/kWh)', f"${sm['usd_per_1m_input_tokens']} / ${sm['usd_per_1m_output_tokens']}"),
    ]
display(pd.DataFrame(rows, columns=['Metric', model_name]).style.hide(axis='index'))

pd.DataFrame([{
    'id': p['id'], 'input tokens': p['input_tokens'], 'output tokens': p['output_tokens'],
    'words': p['words'], 'target': p['words_target'] or '', 'hit limit': 'yes' if p['finish_reason'] == 'length' else '',
    'time (s)': f"{p['total_s']:.1f}", 'decode tok/s': f"{p['decode_tok_per_s']:.1f}",
    'energy (J)': '' if p['energy_j'] is None else f"{p['energy_j']:.1f}",
} for p in result['per_prompt']]).style.hide(axis='index')

Metric,Qwen2.5-14B-Instruct
Prompts,10 (0 hit the 1024-token limit)
Input / output tokens (total),"571 / 6,908"
Total generation time,114.0 s
Average response length,547 words (109% of target)
Prefill speed / decode speed,539 / 61.5 tok/s
Memory per token (KV cache),192 KB
Peak memory (weights + cache),8.57 GB
API cost,$0.00 (runs locally)
"GPU utilisation (avg) / GPU power (avg, peak)","99.3% / 20.3 W, 52.9 W"
"Chip power, avg (CPU + GPU + memory)",39.1 W


id,input tokens,output tokens,words,target,hit limit,time (s),decode tok/s,energy (J)
01,53,417,367,500,,6.9,62.0,262.0
02,61,646,511,500,,10.6,61.7,405.0
03,57,689,567,500,,11.3,61.7,437.3
04,57,641,508,500,,10.6,61.6,410.5
05,87,641,563,500,,10.6,61.4,414.3
06,42,591,500,500,,9.8,61.6,380.1
07,42,869,655,500,,14.3,61.3,564.5
08,65,664,511,500,,11.0,61.5,432.6
09,52,836,607,500,,13.8,61.2,551.4
10,55,914,684,500,,15.1,61.1,603.3


### Simple metrics

The metrics saved for each model, as `<model>_simple_metrics.json` in its output folder: plain names with the unit in every key, grouped into tokens, response length, speed,
memory, energy and cost, plus one short entry per prompt. The hardware blocks cover the whole run: **gpu**, **cpu** (per core cluster), **npu** (Apple Neural Engine), **system_memory** and **whole_chip**.

In [9]:
def simple_metrics(name, result):
    """A flat, readable summary of one model's run: plain names, units in every key."""
    sm, per = result['summary'], result['per_prompt']
    s = SPECS.loc[name]
    size = next(sz for sizes in MODEL_NAME[PLATFORM][TASK].values() for sz, n in sizes.items() if n == name)
    n = len(per)
    words = [p['words'] for p in per]
    targets = [p['words_target'] for p in per if p['words_target']]
    gen_s = sum(p['total_s'] for p in per)
    energy = sm['energy_j']
    measured = energy is not None
    price = sm['electricity_usd_per_kwh']
    r1 = lambda x: None if x is None else round(x, 1)
    return {
        'model': name,
        'family': sm['family'],
        'size': size,
        'repo': sm['repo'],
        'parameters_billion': sm['params_b'],
        'quantization': '4-bit',
        'context_window_tokens': int(s.context),
        'prompt_set': sm['prompt_set'],
        'prompts_run': n,
        'tokens': {
            'input_total': sm['total_input_tokens'],
            'output_total': sm['total_output_tokens'],
            'input_avg_per_prompt': round(sm['total_input_tokens'] / n),
            'output_avg_per_prompt': round(sm['total_output_tokens'] / n),
            'output_limit_per_prompt': MAX_NEW_TOKENS,
        },
        'response_length': {
            'avg_words': round(sum(words) / n),
            'min_words': min(words),
            'max_words': max(words),
            'target_words': max(set(targets), key=targets.count) if targets else None,
            'avg_pct_of_target': sm['avg_words_vs_target_pct'],
            'prompts_within_10pct_of_target': sum(1 for p in per if p['words_target'] and abs(p['words'] / p['words_target'] - 1) <= 0.1),
            'prompts_cut_off_at_token_limit': sm['prompts_hit_token_limit'],
        },
        'speed': {
            'model_load_seconds': sm['load_s'],
            'total_generation_seconds': round(gen_s, 1),
            'avg_seconds_per_prompt': round(gen_s / n, 2),
            'avg_time_to_first_token_ms': round(1000 * sum(p['prefill_s'] for p in per) / n),
            'prompt_reading_tokens_per_second': sm['prefill_tok_per_s'],
            'generation_tokens_per_second': sm['decode_tok_per_s'],
            'ms_per_output_token': round(1000 / sm['decode_tok_per_s'], 2),
            'words_per_second': round(sum(words) / gen_s, 1),
        },
        'memory': {
            'weights_gb': float(s.download_gb),
            'peak_memory_gb': sm['peak_memory_gb'],
            'kv_cache_kb_per_token': sm['kv_kb_per_token'],
            'kv_cache_mb_largest_request': max(p['kv_cache_mb_this_prompt'] for p in per),
            'context_window_used_max_pct': max(p['context_used_pct'] for p in per),
        },
        **(HARDWARE.get(name) or {'hardware': 'not measured (macmon not available)'}),
        'energy': {
            'total_joules': energy,
            'total_watt_hours': round(energy / 3600, 3) if measured else None,
            'avg_joules_per_prompt': round(energy / n, 1) if measured else None,
            'per_input_token_millijoules': sm['prefill_energy_mj_per_input_token'],
            'per_output_token_millijoules': sm['decode_energy_mj_per_output_token'],
            'per_1000_words_joules': round(1000 * energy / sum(words), 1) if measured else None,
        },
        'cost_usd': {
            'api_fee': 0.0,
            'electricity_whole_run': round(energy / 3.6e6 * price, 6) if measured else None,
            'electricity_per_1m_input_tokens': sm['usd_per_1m_input_tokens'],
            'electricity_per_1m_output_tokens': sm['usd_per_1m_output_tokens'],
            'electricity_price_per_kwh': price,
        },
        'per_prompt': [{
            'id': p['id'],
            'words': p['words'],
            'target_words': p['words_target'],
            'input_tokens': p['input_tokens'],
            'output_tokens': p['output_tokens'],
            'seconds': p['total_s'],
            'time_to_first_token_ms': round(1000 * p['prefill_s']),
            'tokens_per_second': p['decode_tok_per_s'],
            'energy_joules': p['energy_j'],
            'cut_off_at_token_limit': p['finish_reason'] == 'length',
        } for p in per],
    }

def save_simple(name, result, folder):
    path = Path(folder) / f'{name}_simple_metrics.json'
    path.write_text(json.dumps(simple_metrics(name, result), indent=2))
    return path

# --- write it for the model run above
print(f'Saved {save_simple(model_name, result, out_dir(model_name))}')
print(json.dumps({k: v for k, v in simple_metrics(model_name, result).items() if k != 'per_prompt'}, indent=2))

Saved outputs/ttt/Qwen2.5/14B/ttt_prompts_500w/Qwen2.5-14B-Instruct_simple_metrics.json
{
  "model": "Qwen2.5-14B-Instruct",
  "family": "Qwen2.5",
  "size": "LARGE",
  "repo": "mlx-community/Qwen2.5-14B-Instruct-4bit",
  "parameters_billion": 14.77,
  "quantization": "4-bit",
  "context_window_tokens": 32768,
  "prompt_set": "ttt_prompts_500w",
  "prompts_run": 10,
  "tokens": {
    "input_total": 571,
    "output_total": 6908,
    "input_avg_per_prompt": 57,
    "output_avg_per_prompt": 691,
    "output_limit_per_prompt": 1024
  },
  "response_length": {
    "avg_words": 547,
    "min_words": 367,
    "max_words": 684,
    "target_words": 500,
    "avg_pct_of_target": 109,
    "prompts_within_10pct_of_target": 4,
    "prompts_cut_off_at_token_limit": 0
  },
  "speed": {
    "model_load_seconds": 1.2,
    "total_generation_seconds": 114.0,
    "avg_seconds_per_prompt": 11.4,
    "avg_time_to_first_token_ms": 104,
    "prompt_reading_tokens_per_second": 539,
    "generation_tokens_per_

## 8. Compare every size in the family

Runs the whole prompt set through SMALL, MEDIUM and LARGE of `FAMILY` (reusing the results already produced
for `SIZE`), unloading each model before the next. Set `TTT_COMPARE=0` (or `RUN_FAMILY_COMPARISON = False`
in section 1) to skip.

In [10]:
import gc

results = {SIZE: (result, runs)}
if RUN_FAMILY_COMPARISON:
    del model, tokenizer
    gc.collect()
    mx.clear_cache()
    for size, name in MODEL_NAME[PLATFORM][TASK][FAMILY].items():
        if size == SIZE:
            continue
        print(f'{size}: {name}')
        m, tok, load_s, _ = load_mlx_model(name)
        r = run_prompt_set(m, tok, name)
        save_responses(name, r)
        res = set_metrics(name, r, load_s)
        save_simple(name, res, out_dir(name))
        results[size] = (res, r)
        del m, tok
        gc.collect()
        mx.clear_cache()

order = [s for s in ('SMALL', 'MEDIUM', 'LARGE') if s in results]
compare = pd.DataFrame({
    s: {
        'Model': sm_['model'],
        'Params (B)': f"{sm_['params_b']:.2f}",
        'Total input / output tokens': f"{sm_['total_input_tokens']:,} / {sm_['total_output_tokens']:,}",
        'Avg words (target %)': f"{sm_['avg_words']}" + (f" ({sm_['avg_words_vs_target_pct']}%)" if sm_['avg_words_vs_target_pct'] else ''),
        'Prompts hitting token limit': sm_['prompts_hit_token_limit'],
        'Total time (s)': sm_['total_time_s'],
        'Prefill (tok/s)': f"{sm_['prefill_tok_per_s']:,}",
        'Decode (tok/s)': sm_['decode_tok_per_s'],
        'Peak memory (GB)': sm_['peak_memory_gb'],
        'GPU util avg (%)': sm_['gpu_util_avg_pct'],
        'GPU power avg (W)': sm_['gpu_power_avg_w'],
        'Energy, whole set (J)': sm_['energy_j'],
        'Energy / input token (mJ)': sm_['prefill_energy_mj_per_input_token'],
        'Energy / output token (mJ)': sm_['decode_energy_mj_per_output_token'],
        '$ / 1M output tokens (electricity)': sm_['usd_per_1m_output_tokens'],
    } for s in order for sm_ in [results[s][0]['summary']]
})
compare.columns.name = FAMILY
compare

Qwen2.5,LARGE
Model,Qwen2.5-14B-Instruct
Params (B),14.77
Total input / output tokens,"571 / 6,908"
Avg words (target %),547 (109%)
Prompts hitting token limit,0
Total time (s),114.0
Prefill (tok/s),539
Decode (tok/s),61.5
Peak memory (GB),8.57
GPU util avg (%),99.277


In [11]:
# Words per response, per prompt and size
pd.DataFrame({s: {r['id']: r['words'] for r in results[s][1]} for s in order}).rename_axis('prompt id')

,LARGE
prompt id,
01,367
02,511
03,567
04,508
05,563
06,500
07,655
08,511
09,607


In [12]:
# The first prompt's response from each size, side by side
for s in order:
    r = results[s][1][0]
    print(f"=== {s}: {results[s][0]['summary']['model']} ({r['words']} words) ===")
    print(textwrap.fill(r['text'][:800], width=100, replace_whitespace=False) + (' ...' if len(r['text']) > 800 else ''), end='\n\n')

=== LARGE: Qwen2.5-14B-Instruct (367 words) ===
Subject: Request for Professional Recommendation for Upcoming Job Opportunity

Dear [Manager's
Name],

I hope this email finds you well. I am writing to request your assistance in providing a
professional recommendation for a job opportunity that I am currently pursuing. As one of my former
managers, I believe your endorsement would greatly enhance my application and provide valuable
insight into my professional capabilities and character.

Over the past [duration of employment], I
have had the privilege of working under your guidance and have learned immensely from your
leadership. Your mentorship has been instrumental in shaping my professional journey and has
significantly contributed to my personal and professional growth. I am now at a stage where I am
seeking to further my career in ...



## 9. All models on this prompt set

Collects every `<model>_simple_metrics.json` produced so far for `PROMPT_SET` (any family, any size, from this
notebook or terminal runs) into `outputs/ttt/all_models_<prompt set>.json` and `.csv`.

In [13]:
FAMILY_ORDER = list(MODEL_NAME[PLATFORM][TASK])
SIZE_ORDER = ['SMALL', 'MEDIUM', 'LARGE']
files = Path('outputs/ttt').glob(f'*/*/{PROMPT_SET}/*_simple_metrics.json')
models = sorted((json.loads(f.read_text()) for f in files),
                key=lambda m: (FAMILY_ORDER.index(m['family']), SIZE_ORDER.index(m['size'])))
overview = [{k: v for k, v in m.items() if k != 'per_prompt'} for m in models]
all_json = Path('outputs/ttt') / f'all_models_{PROMPT_SET}.json'
all_csv = all_json.with_suffix('.csv')
all_json.write_text(json.dumps({'prompt_set': PROMPT_SET, 'models': overview}, indent=2))
flat = pd.json_normalize(overview)
flat.to_csv(all_csv, index=False)
print(f'{len(models)} models -> {all_json} and {all_csv}')

pd.DataFrame({
    'Family': flat['family'], 'Size': flat['size'], 'Model': flat['model'],
    'Params (B)': flat['parameters_billion'],
    'Avg words': flat['response_length.avg_words'],
    'Cut off': flat['response_length.prompts_cut_off_at_token_limit'],
    'Gen tok/s': flat['speed.generation_tokens_per_second'],
    'First token (ms)': flat['speed.avg_time_to_first_token_ms'],
    'Total time (s)': flat['speed.total_generation_seconds'],
    'Peak memory (GB)': flat['memory.peak_memory_gb'],
    'GPU busy (%)': flat.get('gpu.avg_utilization_pct'),
    'GPU power (W)': flat.get('gpu.avg_power_watts'),
    'CPU busy (%)': flat.get('cpu.avg_utilization_pct'),
    'CPU power (W)': flat.get('cpu.avg_power_watts'),
    'NPU power (W)': flat.get('npu.avg_power_watts'),
    'Energy (J)': flat['energy.total_joules'],
    'mJ / output token': flat['energy.per_output_token_millijoules'],
    '$ / 1M output tokens': flat['cost_usd.electricity_per_1m_output_tokens'],
}).style.hide(axis='index')

13 models -> outputs/ttt/all_models_ttt_prompts_500w.json and outputs/ttt/all_models_ttt_prompts_500w.csv


Family,Size,Model,Params (B),Avg words,Cut off,Gen tok/s,First token (ms),Total time (s),Peak memory (GB),GPU busy (%),GPU power (W),CPU busy (%),CPU power (W),NPU power (W),Energy (J),mJ / output token,$ / 1M output tokens
Phi-3,SMALL,Phi-3-mini-128k-instruct,3.820000,684,9,192.200000,48,52.800000,2.720000,98.400000,33.900000,6.100000,5.600000,0.000000,2765.400000,274.909000,0.011450
Phi-3,MEDIUM,Phi-3-small-128k-instruct,7.390000,346,0,107.500000,61,40.700000,4.400000,98.000000,31.000000,5.400000,5.400000,0.000000,2010.500000,463.099000,0.019300
Phi-3,LARGE,Phi-3-medium-128k-instruct,13.960000,667,6,63.100000,95,155.500000,8.200000,99.200000,19.300000,4.900000,4.800000,0.000000,5961.400000,608.329000,0.025350
Llama-3,SMALL,Llama-3.2-1B-Instruct,1.240000,475,0,539.100000,29,12.600000,0.840000,90.900000,21.000000,9.300000,7.100000,0.000000,491.400000,75.975000,0.003170
Llama-3,MEDIUM,Llama-3.2-3B-Instruct,3.210000,466,0,239.300000,41,26.700000,2.070000,95.800000,33.700000,5.600000,5.900000,0.000000,1402.200000,224.402000,0.009350
Llama-3,LARGE,Llama-3.1-8B-Instruct,8.030000,488,0,113.700000,73,57.700000,4.760000,97.900000,21.700000,5.600000,5.000000,0.000000,2329.000000,358.409000,0.014930
Falcon3,SMALL,Falcon3-3B-Instruct,3.230000,496,1,256.900000,39,27.600000,2.050000,95.600000,32.100000,7.100000,5.400000,0.000000,1383.700000,199.784000,0.008320
Falcon3,MEDIUM,Falcon3-7B-Instruct,7.460000,500,0,119.300000,65,58.400000,4.380000,98.100000,25.100000,4.900000,4.700000,0.000000,2525.900000,366.342000,0.015260
Falcon3,LARGE,Falcon3-10B-Instruct,10.310000,450,0,85.000000,92,74.600000,6.040000,98.200000,25.800000,5.200000,4.500000,0.000000,3281.000000,520.996000,0.021710
Qwen2.5,SMALL,Qwen2.5-3B-Instruct,3.090000,552,1,222.500000,46,34.100000,1.960000,96.800000,34.300000,7.500000,6.800000,0.000000,1788.500000,240.445000,0.010020


In [14]:
# Every model's per-prompt metrics (Model_Level_Metrics) in one CSV
prompt_csvs = sorted(Path('metrics_analysis/ttt').glob(f'*/*/{PROMPT_SET}/Model_Level_Metrics/*_prompt_metrics.csv'))
all_prompts = pd.concat([pd.read_csv(p, dtype={'prompt_id': str}) for p in prompt_csvs], ignore_index=True)
all_prompts['family'] = pd.Categorical(all_prompts['family'], FAMILY_ORDER, ordered=True)
all_prompts['size'] = pd.Categorical(all_prompts['size'], SIZE_ORDER, ordered=True)
all_prompts = all_prompts.sort_values(['family', 'size', 'prompt_id'])
combined_csv = Path('metrics_analysis/ttt') / f'all_models_prompt_metrics_{PROMPT_SET}.csv'
all_prompts.to_csv(combined_csv, index=False)
print(f'{len(prompt_csvs)} models, {len(all_prompts)} prompt rows -> {combined_csv}')
all_prompts.groupby(['family', 'size', 'model'], observed=True).agg(
    prompts=('prompt_id', 'count'), input_tokens=('input_tokens', 'sum'), output_tokens=('output_tokens', 'sum'),
    avg_ttft_ms=('time_to_first_token (ms)', 'mean'), avg_inference_rate=('inference_rate (tokens/s)', 'mean'),
    total_inference_time_s=('inference_time (s)', 'sum')).round(1)

13 models, 130 prompt rows -> metrics_analysis/ttt/all_models_prompt_metrics_ttt_prompts_500w.csv


prompts  input_tokens  \
family  size   model                                               
Phi-3   SMALL  Phi-3-mini-128k-instruct         10           351   
        MEDIUM Phi-3-small-128k-instruct        10           326   
        LARGE  Phi-3-medium-128k-instruct       10           351   
Llama-3 SMALL  Llama-3.2-1B-Instruct            10           606   
        MEDIUM Llama-3.2-3B-Instruct            10           606   
        LARGE  Llama-3.1-8B-Instruct            10           606   
Falcon3 SMALL  Falcon3-3B-Instruct              10           391   
        MEDIUM Falcon3-7B-Instruct              10           391   
        LARGE  Falcon3-10B-Instruct             10           391   
Qwen2.5 SMALL  Qwen2.5-3B-Instruct              10           571   
        MEDIUM Qwen2.5-7B-Instruct              10           571   
        LARGE  Qwen2.5-14B-Instruct             10           571   
Llama-2 MEDIUM Llama-2-7b-hf                    10           625   

                                           output_tokens  avg_ttft_ms  \
family  size   model                                                    
Phi-3   SMALL  Phi-3-mini-128k-instruct            10027          NaN   
        MEDIUM Phi-3-small-128k-instruct            4306          NaN   
        LARGE  Phi-3-medium-128k-instruct           9743          NaN   
Llama-3 SMALL  Llama-3.2-1B-Instruct                6384          NaN   
        MEDIUM Llama-3.2-3B-Instruct                6191          NaN   
        LARGE  Llama-3.1-8B-Instruct                6423          NaN   
Falcon3 SMALL  Falcon3-3B-Instruct                  6877          NaN   
        MEDIUM Falcon3-7B-Instruct                  6838          NaN   
        LARGE  Falcon3-10B-Instruct                 6228          NaN   
Qwen2.5 SMALL  Qwen2.5-3B-Instruct                  7374          NaN   
        MEDIUM Qwen2.5-7B-Instruct                  7292          NaN   
        LARGE  Qwen2.5-14B-Instruct                 6908        156.2   
Llama-2 MEDIUM Llama-2-7b-hf                        4720          NaN   

                                           avg_inference_rate  \
family  size   model                                            
Phi-3   SMALL  Phi-3-mini-128k-instruct                   NaN   
        MEDIUM Phi-3-small-128k-instruct                  NaN   
        LARGE  Phi-3-medium-128k-instruct                 NaN   
Llama-3 SMALL  Llama-3.2-1B-Instruct                      NaN   
        MEDIUM Llama-3.2-3B-Instruct                      NaN   
        LARGE  Llama-3.1-8B-Instruct                      NaN   
Falcon3 SMALL  Falcon3-3B-Instruct                        NaN   
        MEDIUM Falcon3-7B-Instruct                        NaN   
        LARGE  Falcon3-10B-Instruct                       NaN   
Qwen2.5 SMALL  Qwen2.5-3B-Instruct                        NaN   
        MEDIUM Qwen2.5-7B-Instruct                        NaN   
        LARGE  Qwen2.5-14B-Instruct                      61.5   
Llama-2 MEDIUM Llama-2-7b-hf                              NaN   

                                           total_inference_time_s  
family  size   model                                               
Phi-3   SMALL  Phi-3-mini-128k-instruct                       0.0  
        MEDIUM Phi-3-small-128k-instruct                      0.0  
        LARGE  Phi-3-medium-128k-instruct                     0.0  
Llama-3 SMALL  Llama-3.2-1B-Instruct                          0.0  
        MEDIUM Llama-3.2-3B-Instruct                          0.0  
        LARGE  Llama-3.1-8B-Instruct                          0.0  
Falcon3 SMALL  Falcon3-3B-Instruct                            0.0  
        MEDIUM Falcon3-7B-Instruct                            0.0  
        LARGE  Falcon3-10B-Instruct                           0.0  
Qwen2.5 SMALL  Qwen2.5-3B-Instruct                            0.0  
        MEDIUM Qwen2.5-7B-Instruct                            0.0  
        LARGE  Qwen2.5-14B-Instruct                         114.0 